In [ ]:
# importar bibliotecas
import pandas as pd
import matplotlib.pyplot as plt

# carregar os dados
df = pd.read_excel("/Users/samuelbucco/Documents/SCTec/Análise de Dados/Etapa Profissionalizar/Mod 1 - Modelagem de Dados/turma-visualizacao-de-dados/alunos/samuel_bucco/semana_04/base_rh.xlsx")

# exibir as primeiras e últimas linhas do dataframe
print(df.head().to_string(index=False))
print()
print(df.tail().to_string(index=False))

In [ ]:
# converter a coluna "Data de Admissão" para o tipo datetime
df["Data_Admissao"] = pd.to_datetime(df["Data_Admissao"], format='%d/%m/%Y', errors="coerce")

# extrair o ano de admissão para uma nova coluna
df["Ano_Admissao"] = df["Data_Admissao"].dt.year

print(df.dtypes.to_string())
print()
print(df[["Data_Admissao", "Ano_Admissao"]].head().to_string(index=False))

In [ ]:
# agrupar os dados por departamento e cargo

agrupado = df.groupby(["Departamento", "Cargo"])["ID_Funcionario"].count()

print(agrupado.to_string())

In [ ]:
# agrupar por ano de admissao e contar admissões por ano e filtrar entre os anos de 2020 a 2024 usando query()

admissoes_ano = df.groupby("Ano_Admissao")["ID_Funcionario"].count().reset_index()
admissoes_ano = admissoes_ano.query("2020 <= Ano_Admissao <= 2024")

print(
    admissoes_ano
    .rename(columns={"ID_Funcionario": "Total_Admissoes"})
    .to_string(index=False)
)

In [ ]:
# criar uma tabela de metas de headcount por departamento com `pd.DataFrame` e fazer `merge left` com o total por departamento. Identificar se a meta foi atingida por departameto e verificar qual departamento atingiu a meta

headcount_real = (
    df.groupby("Departamento")
    .size()
    .reset_index(name="Headcount_Real")
    .sort_values("Departamento")
)

print(headcount_real.to_string(index=False))

# tabela de metas de headcount por departamento
metas = pd.DataFrame({
    "Departamento": ["Financeiro", "RH", "TI", "Vendas", "Produção", "Logística"],
    "Meta_Headcount": [180, 150, 200, 220, 300, 130]
})

print(f"\nTabela de metas de headcount por departamento:")
print(metas.to_string(index=False))

# merge left para juntar o headcount real com as metas
comparativo = pd.merge(headcount_real, metas, on="Departamento", how="left")
print(f"\nComparativo de headcount real e metas:")
print(comparativo.to_string(index=False))

# identificar se a meta foi atingida por departamento
comparativo["Diferença"] = comparativo["Headcount_Real"] - comparativo["Meta_Headcount"]
comparativo["Meta_Atingida"] = comparativo["Diferença"].apply(lambda x: "Sim" if x >= 0 else "Não")
print(f"\nComparativo com indicação de meta atingida:")
print(comparativo.to_string(index=False))


In [ ]:
# criar um pivot table de salario medio por departamento e genero e calcular a dirença entre mulheres e homens

pivot_salario = df.pivot_table(values="Salario", index="Departamento", columns="Genero", aggfunc="mean").round(2)

diferenca_salario = pivot_salario["F"] - pivot_salario["M"]

print(f"\nSalário médio por departamento e gênero:")
for depto in pivot_salario.index:
    print(f"  {depto:12s}: Feminino R$ {pivot_salario.loc[depto, 'F']:,.2f} | Masculino R$ {pivot_salario.loc[depto, 'M']:,.2f}")

print(f"\nDiferença salarial (Feminino - Masculino) por departamento:")
for depto in diferenca_salario.index:
    print(f"  {depto:12s}: R$ {diferenca_salario[depto]:,.2f}")


In [ ]:
#plotar admissões por ano com plt.plot

admissoes_ano_plot = df.groupby("Ano_Admissao")["ID_Funcionario"].count().reset_index().rename(columns={"ID_Funcionario": "Total_Admissoes"})

plt.figure(figsize=(8, 5))
plt.plot(admissoes_ano_plot["Ano_Admissao"], admissoes_ano_plot["Total_Admissoes"], marker="o", linestyle="-", color="steelblue")
plt.title("Admissões por Ano")
plt.xlabel("Ano de Admissão")
plt.ylabel("Total de Admissões")
plt.xticks(admissoes_ano_plot["Ano_Admissao"])
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Plote o pivot_table de salário médio por gênero com pivot.plot(kind="bar")

pivot_salario.plot(kind="bar", figsize=(10, 6), color=["steelblue", "salmon"])
plt.title("Salário Médio por Departamento e Gênero")
plt.xlabel("Departamento")
plt.ylabel("Salário Médio (R$)")
plt.xticks(rotation=45)
plt.legend(title="Gênero")
plt.tight_layout()
plt.show()